# ECCRE | 02: Probability of Default (PD) Scorecard & Calibration Engine

Trains Logistic Scorecard with Platt Scaling to generate institutional-grade 1-year PDs [0, 1].


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.loader import load_raw_data
from src.features.ratios import compute_commodity_ratios
from src.models.pd_engine import train_and_calibrate_pd_model
from src.models.validation import generate_validation_report, compute_calibration_curve_data

%matplotlib inline


### 1. Train Scorecard and Calibrate via Platt Scaling


In [ ]:
df_raw = load_raw_data("data/raw/counterparty_financials.csv")
df_features = compute_commodity_ratios(df_raw)
df_scored, model = train_and_calibrate_pd_model(df_features, save_path="models/pd_model.joblib")
df_scored[["counterparty_name", "vol_adj_icr", "leverage_ratio", "pd_uncalibrated", "pd_calibrated", "target"]].head(10)


### 2. RMG Validation Benchmark Table


In [ ]:
report = generate_validation_report(df_scored["target"], df_scored["pd_calibrated"])
report


### 3. Platt Calibration Curve (Predicted PD vs. Observed Default Frequencies)


In [ ]:
calib_df = compute_calibration_curve_data(df_scored["target"].values, df_scored["pd_calibrated"].values)

plt.figure(figsize=(8, 5))
plt.plot(calib_df["pred_pd"], calib_df["obs_default_rate"], marker='o', label="Platt Calibrated Model", color="crimson")
plt.plot([0, 0.8], [0, 0.8], linestyle="--", color="gray", label="Perfect Calibration")
plt.xlabel("Mean Predicted Probability of Default (PD)")
plt.ylabel("Observed Default Rate")
plt.title("Scorecard Reliability Curve (Calibration Deciles)")
plt.legend()
plt.tight_layout()
plt.show()
